# Attribution and reuse

This notebook documents a data-collection workflow developed by **Sotiris Sideris** for investigative reporting projects led by **[Reporters United](https://www.reportersunited.gr)**, in collaboration with **Investigate Europe** for the **[Fuelling War](https://www.investigate-europe.eu/posts/european-ships-bolster-russian-fossil-fuel-trade-despite-looming-eu-sanctions)** investigation and with the **International Consortium of Investigative Journalists (ICIJ)** for the **[Caspian Cabals](https://www.icij.org/investigations/caspian-cabals/about-caspian-cabals-investigation/)** investigation.


The code and methodology are shared here for transparency and reproducibility.

Parts of the workflow rely on **[Equasis](https://www.equasis.org/EquasisWeb/public/HomePage)**, an access-restricted maritime database, as well as datasets obtained through newsroom collaborations.

Please cite or attribute if reusing this method in reporting or research.


## Scraping vessel management data from Equasis with Selenium

This notebook documents a browser-automation workflow for retrieving vessel and company-management information from Equasis for a predefined list of IMO numbers.

## Reporting context

Equasis is a valuable source for maritime investigations because it links vessels to management and ownership-related entities. In this workflow, Selenium is used to automate the search-and-click process required to retrieve ship particulars and company-management information from the Equasis interface.

## Before you run this notebook

You will need:

- an **Equasis account** with valid login access
- a spreadsheet containing a list of **IMO numbers** to feed into the Equasis search box
- Chrome installed locally, along with a compatible ChromeDriver / Selenium setup

This notebook focuses on the **automation logic** used to retrieve vessel records. It assumes that login access is already available in the browser session or will be completed manually when the Equasis page opens.



## Pipeline overview

1. Load a spreadsheet containing vessel IMO numbers  
2. Open the Equasis search interface in a browser  
3. Use Selenium to submit one IMO number at a time  
4. Open the vessel result page  
5. Extract ship particulars and management/company information  
6. Merge the scraped output back into the source dataset  
7. Export the results for downstream cleaning and review


## 1) Imports and setup

In [ ]:
# Import our working libraries

import pandas as pd
import numpy as np
import glob
import time
import os
import re

from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.common import exceptions
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys

from urllib.request import urlopen
import urllib
import requests
from bs4 import BeautifulSoup

## 2) Display settings

In [ ]:
# Expand dataframe display options for easier inspection during development

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


## 3) Load the IMO input list

This step loads the spreadsheet containing the vessel identifiers that will be searched on Equasis.

For portability, replace the local file path below with the path to your own input file when reproducing the workflow.


In [ ]:
# Load the source spreadsheet and preserve IMO numbers as strings
df = pd.read_excel("/path/to/your_input_file.xlsx", dtype={'IMO (vessel)': object})
df.head()


### Optional sample for testing

During development it can be useful to limit the run to a small number of records before processing the full IMO list.

In [ ]:
# Limit the dataframe to a small sample while testing the workflow
df = df.head()

In [ ]:
# Check the number of rows and columns in the working sample
df.shape

In [ ]:
# Preview the IMO input dataframe
df


## 4) Open Equasis in a browser

This workflow uses **Selenium** because Equasis requires repeated interaction with a live search interface: entering IMO numbers, submitting queries, opening vessel result pages and expanding management panels.

A browser-based approach is useful here because the relevant information is accessed through a sequence of page interactions rather than a simple static URL pattern.

In [ ]:
# Launch a Chrome browser and open the Equasis homepage
driver = webdriver.Chrome()
driver.get('https://www.equasis.org/EquasisWeb/authen/HomePage?fs=HomePage')


## 5) Search Equasis and extract vessel records

The main scraping loop does the following for each IMO number:

- normalizes the identifier to avoid spreadsheet formatting issues
- submits the IMO number into the Equasis search box
- opens the vessel page returned in the search results
- extracts ship particulars
- expands the management section and captures company information

In [ ]:
# Minimal update: fix selectors + imo variable + robust "click IMO result" + robust "open management detail"
# + IMPORTANT: ship particulars now use label-anchored extraction to avoid shifted fields
# No extra debugging / no rate-limit detection




def normalize_imo(x) -> str:
    s = str(x).strip()
    s = re.sub(r"\.0$", "", s)          # Excel float artifact
    digits = re.sub(r"\D", "", s)
    return digits


def value_by_label(driver, label: str) -> str | None:
    """
    Extract value from the ship particulars block by anchoring on the <b> label text.
    This avoids the 'shifted div index' issue you saw on some ships.
    """
    try:
        xp = (
            f"//div[@id='body']//b[normalize-space()='{label}']"
            f"/ancestor::div[contains(@class,'row')][1]"
            f"//div[contains(@class,'col-lg-4') or contains(@class,'col-md-4') or contains(@class,'col-sm-6')][2]"
        )
        return WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, xp))
        ).text.strip()
    except Exception:
        return None


def safe_text(driver, xpath: str, timeout: int = 15) -> str | None:
    """Small helper for the few fields we still take by xpath (flag/callsign/mmsi)."""
    try:
        el = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, xpath))
        )
        txt = (el.text or "").strip()
        return txt if txt else None
    except Exception:
        return None


entries_all = []

for item in df["IMO (vessel)"]:
    all_entries_dict2 = {}

    try:
        imo = normalize_imo(item)

        driver.get("https://www.equasis.org/EquasisWeb/authen/HomePage?fs=HomePage")
        time.sleep(2)

        # Wait for input
        search_box = WebDriverWait(driver, 30).until(
            EC.element_to_be_clickable((By.ID, "P_ENTREE_HOME"))
        )

        # Clear then type
        search_box.click()
        search_box.send_keys(Keys.COMMAND, "a")
        search_box.send_keys(Keys.BACKSPACE)
        search_box.send_keys(imo)

        # Submit
        search_button = WebDriverWait(driver, 30).until(
            EC.element_to_be_clickable((By.XPATH, "//*[@id='searchForm']//button[@type='submit']"))
        )
        driver.execute_script("arguments[0].click();", search_button)

        # Results container
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.ID, "ShipResultId"))
        )

        time.sleep(2)

        # Click the IMO result anchor (Equasis uses onclick submit, not href)
        details_link = WebDriverWait(driver, 30).until(
            EC.presence_of_element_located(
                (By.XPATH,
                 f"//*[@id='ShipResultId']//a[contains(@onclick, \"formShip.P_IMO.value='{imo}'\") "
                 f"or contains(@onclick, \"P_IMO.value='{imo}'\")]")
            )
        )
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", details_link)
        driver.execute_script("arguments[0].click();", details_link)

        # Ensure ship page loaded
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.XPATH, f"//h4[contains(., '{imo}')]"))
        )

        # --- Ship particulars ---
        # Keep the ones that are stable for you
        flag = safe_text(
            driver,
            '//*[@id="body"]/section[3]/div/div/div[1]/div[2]/div[2]/div[1]/div/div/div[1]/div[1]/div[4]'
        )
        if flag: all_entries_dict2["Flag_RU"] = flag

        call_sign = safe_text(
            driver,
            '//*[@id="body"]/section[3]/div/div/div[1]/div[2]/div[2]/div[1]/div/div/div[1]/div[2]/div[2]'
        )
        if call_sign: all_entries_dict2["Call_Sign_RU"] = call_sign

        mmsi = safe_text(
            driver,
            '//*[@id="body"]/section[3]/div/div/div[1]/div[2]/div[2]/div[1]/div/div/div[1]/div[3]/div[2]'
        )
        if mmsi: all_entries_dict2["MMSI_RU"] = mmsi

        # These are the ones that were shifting: extract by LABEL instead
        gt = value_by_label(driver, "Gross tonnage")
        if gt: all_entries_dict2["Gross_Tonnage_RU"] = gt

        dwt = value_by_label(driver, "DWT")
        if dwt: all_entries_dict2["DWT_RU"] = dwt

        typ = value_by_label(driver, "Type of ship")
        if typ: all_entries_dict2["Type_RU"] = typ

        yb = value_by_label(driver, "Year of build")
        if yb: all_entries_dict2["Year_Build_RU"] = yb

        st = value_by_label(driver, "Status")
        if st: all_entries_dict2["Status_RU"] = st

        time.sleep(2)

        # --- Management detail: open the collapse ---
        management_toggle = WebDriverWait(driver, 30).until(
            EC.element_to_be_clickable((By.XPATH, "//a[contains(@class,'boutonCollapse') and @href='#collapse3']"))
        )
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", management_toggle)
        driver.execute_script("arguments[0].click();", management_toggle)

        # Wait for management content (mobile blocks or table)
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located(
                (By.XPATH, "//*[@id='collapse3']//*[contains(@class,'blocLSMobile') or self::table]")
            )
        )

        doc = BeautifulSoup(driver.page_source, "html.parser")
        blocks = doc.find_all("div", class_="blocLSMobile col-xs-12 col-sm-12")

        if len(blocks) > 0:
            all_entries_dict2["First_Company_RU"] = blocks[0].get_text(" ", strip=True)
        if len(blocks) > 1:
            all_entries_dict2["Second_Company_RU"] = blocks[1].get_text(" ", strip=True)
        if len(blocks) > 2:
            all_entries_dict2["Third_Company_RU"] = blocks[2].get_text(" ", strip=True)

        all_entries_dict2["IMO2_RU"] = imo

    except Exception:
        all_entries_dict2["IMO2_RU"] = str(item).strip()

    entries_all.append(all_entries_dict2)
    print(all_entries_dict2)
    print("-----")
    time.sleep(5)


## 6) Build a dataframe from the scraped records

In [ ]:
# Convert the collected vessel dictionaries into a dataframe
an_df = pd.DataFrame(entries_all)


In [ ]:
# Check the size of the scraped output dataframe
an_df.shape


## 7) Merge the scraped Equasis data back into the source dataset

The scraped dataframe is merged back into the original spreadsheet so that the newly collected vessel and management information can be reviewed alongside the source fields.


In [ ]:
# Merge the source dataframe with the scraped Equasis output by row order
merged_df = pd.merge(df, an_df, left_index=True, right_index=True)

In [ ]:
# Check the size of the merged dataframe
merged_df.shape

In [ ]:
# Preview the merged dataframe
merged_df

## 8) Prepare an export copy

In [ ]:
# Create a copy of the merged dataframe for final review and export
NEWdf = merged_df.copy()


# --------------------------------------------------
# 1) Parse single Equasis company block
# --------------------------------------------------

def parse_company_block(s: str):
    if pd.isna(s) or not str(s).strip():
        return {
            "name": np.nan,
            "imo": np.nan,
            "role": np.nan,
            "address": np.nan,
            "date_since": np.nan,
        }

    s = str(s)
    s = re.sub(r"[\n\t]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Main pattern
    m = re.search(
        r"^(?P<name>.*?)\s+IMO number\s+(?P<imo>\d+)\s+"
        r"Role\s+(?P<role>.*?)\s+"
        r"Address\s+(?P<address>.*?)\s+"
        r"Date of effect since\s+(?P<date_since>\d{2}/\d{2}/\d{4})\s*$",
        s,
        flags=re.IGNORECASE,
    )

    # Fallback (looser)
    if not m:
        m = re.search(
            r"^(?P<name>.*?)\s+IMO number\s+(?P<imo>\d+)\s+"
            r"Role\s+(?P<role>.*?)\s+"
            r"Address\s+(?P<address>.*?)(?:\s+Date of effect since\s+(?P<date_since>.*))?$",
            s,
            flags=re.IGNORECASE,
        )

    if not m:
        return {
            "name": np.nan,
            "imo": np.nan,
            "role": np.nan,
            "address": np.nan,
            "date_since": np.nan,
        }

    d = m.groupdict()

    for k in d:
        if d[k] is not None:
            d[k] = re.sub(r"\s+", " ", str(d[k])).strip()
        else:
            d[k] = np.nan

    return d


# --------------------------------------------------
# 2) Expand First/Second/Third into structured cols
# --------------------------------------------------

def expand_company(df, src, prefix):

    parsed = df[src].apply(parse_company_block).apply(pd.Series)

    parsed = parsed.rename(
        columns={
            "name": f"{prefix}_NAME",
            "imo": f"{prefix}_IMO",
            "role": f"{prefix}_ROLE",
            "address": f"{prefix}_ADDRESS",
            "date_since": f"{prefix}_DATE_SINCE",
        }
    )

    return pd.concat([df, parsed], axis=1)


NEWdf = expand_company(NEWdf, "First_Company_RU", "C1")
NEWdf = expand_company(NEWdf, "Second_Company_RU", "C2")
NEWdf = expand_company(NEWdf, "Third_Company_RU", "C3")


# --------------------------------------------------
# 3) Pretty formatted versions (optional)
# --------------------------------------------------

def pretty(prefix):
    return (
        "Company Name: " + NEWdf[f"{prefix}_NAME"].fillna("") +
        " | IMO number: " + NEWdf[f"{prefix}_IMO"].fillna("") +
        " | Role: " + NEWdf[f"{prefix}_ROLE"].fillna("") +
        " | Address: " + NEWdf[f"{prefix}_ADDRESS"].fillna("") +
        " | Date of effect since: " + NEWdf[f"{prefix}_DATE_SINCE"].fillna("")
    ).replace(r"\s+\|\s+\|", " | ", regex=True).str.strip()


NEWdf["First_Company_FMT"]  = pretty("C1")
NEWdf["Second_Company_FMT"] = pretty("C2")
NEWdf["Third_Company_FMT"]  = pretty("C3")


# --------------------------------------------------
# 4) Pick company by role
# --------------------------------------------------

def pick_by_role(df, role, field):

    out = pd.Series(np.nan, index=df.index, dtype="object")

    for p in ["C1", "C2", "C3"]:

        mask = df[f"{p}_ROLE"].fillna("").str.contains(
            role, case=False, regex=False
        )

        out = out.where(~mask, df[f"{p}_{field}"])

    return out


ROLES = {
    "Ship manager/Commercial manager": "Ship_Manager_Commercial_Manager",
    "Registered owner": "Registered_Owner",
    "ISM Manager": "ISM_Manager",
}


for role_txt, col_base in ROLES.items():

    NEWdf[f"{col_base}_RU_NAME"] = pick_by_role(NEWdf, role_txt, "NAME")
    NEWdf[f"{col_base}_RU_IMO"] = pick_by_role(NEWdf, role_txt, "IMO")
    NEWdf[f"{col_base}_RU_ADDRESS"] = pick_by_role(NEWdf, role_txt, "ADDRESS")
    NEWdf[f"{col_base}_RU_DATE"] = pick_by_role(NEWdf, role_txt, "DATE_SINCE")


# --------------------------------------------------
# 5) Country extraction (robust)
# --------------------------------------------------

def extract_country(addr):

    if pd.isna(addr) or not str(addr).strip():
        return np.nan

    parts = [p.strip() for p in str(addr).split(",") if p.strip()]

    if not parts:
        return np.nan

    return parts[-1].rstrip(".")


for col_base in ROLES.values():

    addr_col = f"{col_base}_RU_ADDRESS"

    NEWdf[f"{col_base}_RU_COUNTRY"] = NEWdf[addr_col].apply(extract_country)


In [ ]:
# Preview the final export dataframe
NEWdf

In [ ]:
# Export the enriched Equasis dataset to CSV

NEWdf.to_csv("equasis_vessel_management_dataset.csv", index=False)


## 9) Export results

The final dataframe is exported to CSV for downstream cleaning, validation and entity-resolution work in tools such as OpenRefine.



## Output dataset

The final output of this notebook is:

`equasis_vessel_management_dataset.csv`

This file contains:

- the original vessel dataset with IMO identifiers
- enriched vessel attributes extracted from Equasis (flag, tonnage, ship type, year built, etc.)
- management company information retrieved from the vessel management section

The exported dataset is designed to be used in downstream investigative workflows such as:

- entity resolution
- company network mapping
- corporate ownership analysis
- further cleaning and structuring in OpenRefine or Python



## Notes on reproducibility

- Replace the placeholder input file path with your own spreadsheet location.
- The notebook assumes access to an authenticated Equasis session.
- Because Equasis is an interactive website, Selenium is used to replicate the sequence of manual actions required to search by IMO number and expand management details.
- If the site layout changes, selectors may need to be updated.
